# Day 29 - Segment tree and Fenwick tree: range queries under updates

Code: [day 29 on GitHub](https://github.com/KingRei/100DaysPython/tree/master/day%2029%20-%20segment%20tree%20and%20fenwick)

Two operations on an array: write one cell, and ask for the sum of a slice.
Either one alone is trivial. Wanting both to be fast at the same time is the
entire subject of today, and it has two standard answers - a **segment tree**,
which cuts a range into whole subtrees, and a **Fenwick tree** (also called a
**binary indexed tree**; they are the same structure), which throws the tree
away and keeps only what the binary representation of the index implies.

## 1. The two naive answers are mirror images

A plain array updates in `O(1)` and sums in `O(n)`. A prefix-sum array sums in
`O(1)` and updates in `O(n)`, because writing `a[i]` invalidates every prefix
after `i`. Neither is a compromise - each is perfect at one job and useless at
the other.

In [1]:
import random
import time


class PlainArray:
    """O(1) update, O(n) query."""

    def __init__(self, data):
        self.a = list(data)
        self.query_work = 0

    def update(self, i, value):
        self.a[i] = value

    def range_sum(self, l, r):                       # sum of a[l:r]
        self.query_work += r - l
        return sum(self.a[l:r])

class PrefixSum:
    """O(1) query, O(n) update - the mirror image."""

    def __init__(self, data):
        self.a = list(data)
        self.update_work = 0
        self._rebuild()

    def _rebuild(self):
        p = [0] * (len(self.a) + 1)
        for i, x in enumerate(self.a):
            p[i + 1] = p[i] + x
        self.p = p

    def update(self, i, value):
        self.a[i] = value
        self.update_work += len(self.a)              # every later prefix moves
        self._rebuild()

    def range_sum(self, l, r):
        return self.p[r] - self.p[l]


a = PlainArray([3, 1, 4, 1, 5, 9, 2, 6])
p = PrefixSum([3, 1, 4, 1, 5, 9, 2, 6])
print('plain  range_sum(1, 7) =', a.range_sum(1, 7), ' work', a.query_work)
print('prefix range_sum(1, 7) =', p.range_sum(1, 7))
p.update(2, 7)
print('prefix update(2, 7) rewrote', p.update_work, 'cells of', len(p.a))

plain  range_sum(1, 7) = 22  work 6
prefix range_sum(1, 7) = 22
prefix update(2, 7) rewrote 8 cells of 8


## 2. The segment tree: a range is a handful of whole subtrees

Store the array in the second half of a flat array of size `2n`: leaf `i` lives
at `t[n + i]`, and every internal node `k` holds `t[2k] + t[2k+1]`. Walking up
from a leaf is `k >>= 1`; there are no pointers and no recursion.

A query starts at both ends of the range and climbs. Whenever the left cursor
is a right child it is the *only* part of its parent that belongs to the range,
so its value is taken and the cursor moves on; symmetrically on the right.

In [2]:
class SegmentTree:
    """Point update, range query, over any associative combine().

    Layout: a flat array of size 2n.  Leaf i lives at t[n + i], and the
    parent of node k is k >> 1.  No recursion, no 4n allocation, and the
    tree is built in one backwards pass.
    """

    def __init__(self, data, combine=None, identity=0):
        self.n = len(data)
        self.combine = combine or (lambda a, b: a + b)
        self.identity = identity
        self.t = [identity] * (2 * self.n)
        self.t[self.n:] = list(data)                 # leaves
        for k in range(self.n - 1, 0, -1):           # internal nodes, bottom up
            self.t[k] = self.combine(self.t[2 * k], self.t[2 * k + 1])
        self.nodes_touched = 0

    def update(self, i, value):
        """Set a[i] = value and repair the O(log n) ancestors."""
        k = i + self.n
        self.t[k] = value
        k >>= 1
        while k:
            self.nodes_touched += 1
            self.t[k] = self.combine(self.t[2 * k], self.t[2 * k + 1])
            k >>= 1

    def query(self, l, r):
        """Combine over a[l:r] - half open, like a Python slice."""
        res_l, res_r = self.identity, self.identity
        l += self.n
        r += self.n
        while l < r:
            self.nodes_touched += 1
            if l & 1:                                # l is a right child: take it
                res_l = self.combine(res_l, self.t[l])
                l += 1
            if r & 1:                                # r is a right child: take r-1
                r -= 1
                res_r = self.combine(self.t[r], res_r)
            l >>= 1
            r >>= 1
        return self.combine(res_l, res_r)


t = SegmentTree([1, 3, 5, 7, 9, 11])
print('query(1, 5) =', t.query(1, 5), ' (3 + 5 + 7 + 9)')
t.update(2, 100)
print('after update(2, 100):', t.query(1, 5))

query(1, 5) = 24  (3 + 5 + 7 + 9)
after update(2, 100): 119


### Which nodes does a query actually visit?

`canonical_cover` records them. The answer is never more than **two nodes per
level**: on any level at most one node can be a partial left edge and one a
partial right edge, everything between them is whole. That is where the
`2 * log2(n)` bound comes from.

In [3]:
def canonical_cover(n, l, r):
    """The exact set of tree nodes an iterative query [l, r) visits.

    This is the thing worth seeing: an arbitrary range is cut into at most
    2*ceil(log2(n)) whole subtrees, and never more.
    """
    out = []
    l += n
    r += n
    while l < r:
        if l & 1:
            out.append(l)
            l += 1
        if r & 1:
            r -= 1
            out.append(r)
        l >>= 1
        r >>= 1
    return sorted(out)

def query_cost_profile(n, trials=20000, seed=29):
    """Average and worst node count for a segment tree query on [0, n)."""
    rng = random.Random(seed)
    tot = worst = 0
    for _ in range(trials):
        l = rng.randrange(n)
        r = rng.randrange(l + 1, n + 1)
        c = len(canonical_cover(n, l, r))
        tot += c
        worst = max(worst, c)
    return tot / trials, worst


print('n = 16, query [3, 13) ->', canonical_cover(16, 3, 13))
for n in (1024, 65536, 1000000):
    avg, worst = query_cost_profile(n)
    print('n = %-9d average %5.2f nodes, worst seen %d' % (n, avg, worst))

n = 16, query [3, 13) -> [5, 6, 19, 28]
n = 1024      average  7.27 nodes, worst seen 16
n = 65536     average 13.21 nodes, worst seen 25
n = 1000000   average 17.01 nodes, worst seen 30


## 3. Lazy propagation: stop at the node that is fully covered

Adding `delta` to a whole range does not require touching the leaves. A node
whose range lies entirely inside the update already knows its new sum -
`delta * width` - so we apply that and leave a note (`lazy`) saying "my children
have not heard about this yet". The note is pushed down only when somebody
actually looks inside.

In [4]:
class LazySegmentTree:
    """Range add, range sum.  Recursive, 4n nodes, one pending value per node.

    The idea: when a whole node is covered by the update, do NOT descend.
    Apply the change to that node's aggregate and leave a note ("lazy") on
    it.  The note is pushed down only if someone later needs to look inside.
    """

    def __init__(self, data):
        self.n = len(data)
        self.sum = [0] * (4 * self.n)
        self.lazy = [0] * (4 * self.n)
        self.pushes = 0
        self.visits = 0
        if self.n:
            self._build(1, 0, self.n - 1, list(data))

    def _build(self, node, lo, hi, data):
        if lo == hi:
            self.sum[node] = data[lo]
            return
        mid = (lo + hi) // 2
        self._build(2 * node, lo, mid, data)
        self._build(2 * node + 1, mid + 1, hi, data)
        self.sum[node] = self.sum[2 * node] + self.sum[2 * node + 1]

    def _apply(self, node, lo, hi, delta):
        self.sum[node] += delta * (hi - lo + 1)      # +delta on each element
        self.lazy[node] += delta                     # note for the children

    def _push(self, node, lo, hi):
        if self.lazy[node]:
            mid = (lo + hi) // 2
            self._apply(2 * node, lo, mid, self.lazy[node])
            self._apply(2 * node + 1, mid + 1, hi, self.lazy[node])
            self.lazy[node] = 0
            self.pushes += 1

    def range_add(self, l, r, delta):                # inclusive [l, r]
        self._range_add(1, 0, self.n - 1, l, r, delta)

    def _range_add(self, node, lo, hi, l, r, delta):
        self.visits += 1
        if r < lo or hi < l:                         # disjoint
            return
        if l <= lo and hi <= r:                      # fully covered: stop here
            self._apply(node, lo, hi, delta)
            return
        self._push(node, lo, hi)
        mid = (lo + hi) // 2
        self._range_add(2 * node, lo, mid, l, r, delta)
        self._range_add(2 * node + 1, mid + 1, hi, l, r, delta)
        self.sum[node] = self.sum[2 * node] + self.sum[2 * node + 1]

    def range_sum(self, l, r):                       # inclusive [l, r]
        return self._range_sum(1, 0, self.n - 1, l, r)

    def _range_sum(self, node, lo, hi, l, r):
        self.visits += 1
        if r < lo or hi < l:
            return 0
        if l <= lo and hi <= r:
            return self.sum[node]
        self._push(node, lo, hi)
        mid = (lo + hi) // 2
        return (self._range_sum(2 * node, lo, mid, l, r)
                + self._range_sum(2 * node + 1, mid + 1, hi, l, r))


z = LazySegmentTree([0] * 10)
z.range_add(2, 5, 5)
z.range_add(4, 7, 3)
print('range_sum(0, 9) =', z.range_sum(0, 9))
print('range_sum(4, 5) =', z.range_sum(4, 5), ' visits', z.visits, ' pushes', z.pushes)

range_sum(0, 9) = 32
range_sum(4, 5) = 16  visits 36  pushes 3


In [5]:
class EagerRangeAddTree:
    """The same range-add, but without lazy: descend all the way to the leaves.

    Kept only to measure how much lazy propagation actually saves.
    """

    def __init__(self, data):
        self.n = len(data)
        self.sum = [0] * (4 * self.n)
        self.visits = 0
        if self.n:
            self._build(1, 0, self.n - 1, list(data))

    def _build(self, node, lo, hi, data):
        if lo == hi:
            self.sum[node] = data[lo]
            return
        mid = (lo + hi) // 2
        self._build(2 * node, lo, mid, data)
        self._build(2 * node + 1, mid + 1, hi, data)
        self.sum[node] = self.sum[2 * node] + self.sum[2 * node + 1]

    def range_add(self, l, r, delta):
        self._add(1, 0, self.n - 1, l, r, delta)

    def _add(self, node, lo, hi, l, r, delta):
        self.visits += 1
        if r < lo or hi < l:
            return
        if lo == hi:
            self.sum[node] += delta
            return
        mid = (lo + hi) // 2
        self._add(2 * node, lo, mid, l, r, delta)
        self._add(2 * node + 1, mid + 1, hi, l, r, delta)
        self.sum[node] = self.sum[2 * node] + self.sum[2 * node + 1]

    def range_sum(self, l, r):
        return self._sum(1, 0, self.n - 1, l, r)

    def _sum(self, node, lo, hi, l, r):
        if r < lo or hi < l:
            return 0
        if l <= lo and hi <= r:
            return self.sum[node]
        mid = (lo + hi) // 2
        return (self._sum(2 * node, lo, mid, l, r)
                + self._sum(2 * node + 1, mid + 1, hi, l, r))

def lazy_vs_eager(n=100000, ops=2000, seed=29):
    rng = random.Random(seed)
    data = [0] * n
    lz = LazySegmentTree(data)
    eg = EagerRangeAddTree(data)
    for _ in range(ops):
        l = rng.randrange(n)
        r = rng.randrange(l, n)
        v = rng.randint(1, 9)
        lz.range_add(l, r, v)
        eg.range_add(l, r, v)
    assert lz.range_sum(0, n - 1) == eg.range_sum(0, n - 1)
    return lz.visits, eg.visits, lz.pushes


lazy_visits, eager_visits, pushes = lazy_vs_eager()
print('n = 100000, 2000 random range-adds')
print('  eager (down to every leaf) : %d node visits' % eager_visits)
print('  lazy  (stop when covered)  : %d node visits' % lazy_visits)
print('  speedup %.0fx, with %d pushes down' % (eager_visits / lazy_visits, pushes))

n = 100000, 2000 random range-adds
  eager (down to every leaf) : 100775266 node visits
  lazy  (stop when covered)  : 116963 node visits
  speedup 862x, with 42681 pushes down


## 4. The Fenwick tree: the shape is in the bits

`i & -i` is the lowest set bit of `i`. A Fenwick tree stores, at `t[i]`, the sum
of exactly that many elements ending at `i`. Nothing else is stored - no
children, no ranges, no tree array. `n` counters cover every prefix.

Reading a prefix peels set bits off the index (`i -= i & -i`); writing a cell
adds them (`i += i & -i`). Both loops run `popcount`-many times, so at most
`log2(n)` steps.

In [6]:
class Fenwick:
    """1-based BIT.  Point add, prefix sum, and a binary-lifting search."""

    def __init__(self, n_or_data):
        if isinstance(n_or_data, int):
            self.n = n_or_data
            self.t = [0] * (self.n + 1)
        else:
            data = list(n_or_data)
            self.n = len(data)
            self.t = [0] + data                      # t[i] starts as a[i-1]
            # O(n) build: push each cell into its parent once.
            for i in range(1, self.n + 1):
                j = i + (i & -i)
                if j <= self.n:
                    self.t[j] += self.t[i]
        self.steps = 0

    def add(self, i, delta):
        """a[i] += delta, with i 0-based on the outside."""
        i += 1
        while i <= self.n:
            self.steps += 1
            self.t[i] += delta
            i += i & -i                              # go to the parent

    def prefix(self, i):
        """Sum of a[0:i] - i elements, 0-based half-open."""
        s = 0
        while i > 0:
            self.steps += 1
            s += self.t[i]
            i -= i & -i                              # drop the lowest set bit
        return s

    def range_sum(self, l, r):                       # a[l:r]
        return self.prefix(r) - self.prefix(l)

    def lower_bound(self, target):
        """Smallest 0-based i with prefix(i+1) >= target, for non-negative a.

        Binary lifting over the tree, in O(log n) - and it needs no extra
        memory, because the powers of two ARE the tree.  This is the trick
        that makes a BIT a usable ordered multiset over a small key space.
        """
        pos = 0
        bit = 1 << (self.n.bit_length())
        while bit:
            nxt = pos + bit
            if nxt <= self.n and self.t[nxt] < target:
                pos = nxt
                target -= self.t[nxt]
            bit >>= 1
        return pos                                   # 0-based index

    def covered_by(self, i):
        """Which elements t[i] holds - purely for the explanation."""
        return (i - (i & -i), i)                     # half-open, 0-based


f = Fenwick(list(range(1, 17)))
for i in (1, 2, 4, 6, 8, 12, 16):
    lo, hi = f.covered_by(i)
    print('t[%2d]  binary %5s  lowbit %2d  covers a[%2d:%2d]'
          % (i, format(i, 'b'), i & -i, lo, hi))

t[ 1]  binary     1  lowbit  1  covers a[ 0: 1]
t[ 2]  binary    10  lowbit  2  covers a[ 0: 2]
t[ 4]  binary   100  lowbit  4  covers a[ 0: 4]
t[ 6]  binary   110  lowbit  2  covers a[ 4: 6]
t[ 8]  binary  1000  lowbit  8  covers a[ 0: 8]
t[12]  binary  1100  lowbit  4  covers a[ 8:12]
t[16]  binary 10000  lowbit 16  covers a[ 0:16]


In [7]:
print('range_sum(3, 11) =', f.range_sum(3, 11), ' (4 + 5 + ... + 11)')

i = 13
walk = []
while i > 0:
    walk.append(i)
    i -= i & -i
print('prefix(13) reads t' + ' + t'.join(str(w) for w in walk),
      '=', f.prefix(13), ' (%d steps = popcount(13))' % len(walk))

i, up = 5, []
while i <= 16:
    up.append(i)
    i += i & -i
print('add(index 4, v) updates t', up, sep='')

print('lower_bound(40) =', f.lower_bound(40),
      ' prefix there =', f.prefix(f.lower_bound(40) + 1))

range_sum(3, 11) = 60  (4 + 5 + ... + 11)
prefix(13) reads t13 + t12 + t8 = 91  (3 steps = popcount(13))
add(index 4, v) updates t[5, 6, 8, 16]
lower_bound(40) = 8  prefix there = 45


### Range updates without a segment tree

One BIT over a *difference* array gives range-add plus point-query. Two BITs
give range-add plus range-sum, using
`prefix(i) = B1.prefix(i) * i - B2.prefix(i)`. Same asymptotics as a lazy
segment tree, a quarter of the memory - and it only works because addition can
be undone.

In [8]:
class FenwickRangeUpdate:
    """Range add + point query, using one BIT over the difference array.

    add(l, r, v) becomes d[l] += v, d[r+1] -= v, and a[i] is prefix(d, i).
    """

    def __init__(self, n):
        self.n = n
        self.d = Fenwick(n + 1)

    def range_add(self, l, r, v):                    # inclusive [l, r]
        self.d.add(l, v)
        self.d.add(r + 1, -v)

    def point_query(self, i):
        return self.d.prefix(i + 1)

class FenwickRangeRange:
    """Range add AND range sum, with two BITs.

    prefix(i) of the updated array is  B1.prefix(i)*i - B2.prefix(i),
    which is the standard two-BIT trick.  Half the memory of a lazy
    segment tree and a much shorter inner loop, but it only works because
    addition has an inverse.
    """

    def __init__(self, n):
        self.n = n
        self.b1 = Fenwick(n + 2)
        self.b2 = Fenwick(n + 2)

    def range_add(self, l, r, v):                    # inclusive [l, r]
        self.b1.add(l, v)
        self.b1.add(r + 1, -v)
        self.b2.add(l, v * l)                    # v * (L-1) with L = l+1
        self.b2.add(r + 1, -v * (r + 1))         # -v * R  with R = r+1

    def _prefix(self, i):                            # sum of a[0:i]
        return self.b1.prefix(i) * i - self.b2.prefix(i)

    def range_sum(self, l, r):                       # inclusive [l, r]
        return self._prefix(r + 1) - self._prefix(l)


d = FenwickRangeUpdate(10)
d.range_add(2, 6, 5)
d.range_add(4, 8, 3)
print('range add + point query:', [d.point_query(i) for i in range(10)])

rr = FenwickRangeRange(10)
rr.range_add(2, 6, 5)
rr.range_add(4, 8, 3)
print('range add + range sum  : sum of a[0..9] =', rr.range_sum(0, 9),
      ' sum of a[3..6] =', rr.range_sum(3, 6))

range add + point query: [0, 0, 5, 5, 8, 8, 8, 3, 3, 0]
range add + range sum  : sum of a[0..9] = 40  sum of a[3..6] = 29


## 5. What a Fenwick tree cannot do

`range(l, r) = prefix(r) - prefix(l)` needs an **inverse**. Sum has one,
xor has one, `min` does not. A prefix-min BIT can only ever let values shrink,
so after `a[3] = 9` it still reports the `1` that used to be there.

A segment tree never subtracts - it only combines disjoint pieces - so any
associative operation works: sum, min, max, gcd, matrix product.

In [9]:
class MinSegmentTree(SegmentTree):
    """The same class with a different combine - that is the entire change."""

    def __init__(self, data):
        super().__init__(data, combine=min, identity=float('inf'))

class BrokenMinBIT:
    """A prefix-min BIT.  It works right up until a value increases.

    Kept in the module deliberately: this is the bug people actually write,
    and it passes every test where values only ever go down.
    """

    def __init__(self, n):
        self.n = n
        self.t = [float('inf')] * (n + 1)

    def update(self, i, value):                      # "a[i] = value"
        i += 1
        while i <= self.n:
            self.t[i] = min(self.t[i], value)        # can only ever shrink
            i += i & -i

    def prefix_min(self, i):                         # min of a[0:i]
        m = float('inf')
        while i > 0:
            m = min(m, self.t[i])
            i -= i & -i
        return m

def min_bit_failure():
    """Show the failure concretely.  Returns (true, broken BIT, segment tree)."""
    a = [5, 3, 8, 1]
    bit = BrokenMinBIT(len(a))
    for i, x in enumerate(a):
        bit.update(i, x)
    a[3] = 9                                          # the 1 goes away
    bit.update(3, 9)
    seg = MinSegmentTree([5, 3, 8, 1])
    seg.update(3, 9)
    return min(a), bit.prefix_min(4), seg.query(0, 4)


true_min, bit_min, seg_min = min_bit_failure()
print('a = [5, 3, 8, 1], then a[3] = 9')
print('  real min        =', true_min)
print('  prefix-min BIT  =', bit_min, '  <- wrong, it never forgot the 1')
print('  min segment tree=', seg_min)

a = [5, 3, 8, 1], then a[3] = 9
  real min        = 3
  prefix-min BIT  = 1   <- wrong, it never forgot the 1
  min segment tree= 3


## 6. Cost, side by side

Same workload, four structures. The prefix-sum array is the cautionary tale:
unbeatable at queries, catastrophic once writes are mixed in.

In [10]:
def timing(n=20000, ops=2000, seed=29):
    """Wall clock for a mixed update/query workload.  Ratios, not absolutes."""
    rng = random.Random(seed)
    data = [rng.randint(0, 99) for _ in range(n)]
    plan = []
    for _ in range(ops):
        i = rng.randrange(n)
        v = rng.randint(0, 99)
        l = rng.randrange(n)
        r = rng.randrange(l + 1, n + 1)
        plan.append((i, v, l, r))

    out = {}

    a = PlainArray(data)
    t0 = time.perf_counter()
    for i, v, l, r in plan:
        a.update(i, v)
        a.range_sum(l, r)
    out['plain array'] = time.perf_counter() - t0

    p = PrefixSum(data)
    t0 = time.perf_counter()
    for i, v, l, r in plan:
        p.update(i, v)
        p.range_sum(l, r)
    out['prefix sum'] = time.perf_counter() - t0

    s = SegmentTree(data)
    t0 = time.perf_counter()
    for i, v, l, r in plan:
        s.update(i, v)
        s.query(l, r)
    out['segment tree'] = time.perf_counter() - t0

    cur = list(data)
    f = Fenwick(data)
    t0 = time.perf_counter()
    for i, v, l, r in plan:
        f.add(i, v - cur[i])
        cur[i] = v
        f.range_sum(l, r)
    out['fenwick'] = time.perf_counter() - t0
    return out


t_ = timing()
base = t_['fenwick']
for k in ('plain array', 'prefix sum', 'segment tree', 'fenwick'):
    print('  %-14s %8.1f ms  %6.1fx fenwick' % (k, t_[k] * 1000, t_[k] / base))

  plain array        51.6 ms     3.7x fenwick
  prefix sum       4598.1 ms   332.2x fenwick
  segment tree       22.2 ms     1.6x fenwick
  fenwick            13.8 ms     1.0x fenwick


## 7. LeetCode

**307. Range Sum Query - Mutable** is the canonical problem for today, built so
that neither naive answer survives. Note the BIT detail: a BIT stores deltas, so
a *set* becomes `add(i, new - old)` and you must keep the old values yourself.

**315. Count of Smaller Numbers After Self** is the trick worth remembering -
scan right to left and use a BIT over compressed ranks as a running histogram,
so "how many smaller values have I already seen" is one prefix query.

**370. Range Addition** is the anticlimax, and the point of including it: every
update comes before every query, so a difference array answers it in `O(n + k)`
with no tree at all. A tree earns its keep only when updates and queries are
interleaved.

In [11]:
class NumArrayBIT:
    """LC 307 - Range Sum Query, Mutable.  The Fenwick answer."""

    def __init__(self, nums):
        self.a = list(nums)
        self.bit = Fenwick(self.a)                   # O(n) build

    def update(self, index, val):
        self.bit.add(index, val - self.a[index])     # BITs add, they do not set
        self.a[index] = val

    def sumRange(self, left, right):                 # inclusive
        return self.bit.range_sum(left, right + 1)

class NumArraySeg:
    """LC 307 - the segment tree answer.  Same complexity, more memory."""

    def __init__(self, nums):
        self.tree = SegmentTree(nums)

    def update(self, index, val):
        self.tree.update(index, val)

    def sumRange(self, left, right):
        return self.tree.query(left, right + 1)

def count_smaller(nums):
    """LC 315 - Count of Smaller Numbers After Self.

    Scan right to left; for each value ask "how many already-seen values are
    strictly smaller".  That is a prefix sum over a frequency array, and the
    frequency array is being updated as we go - exactly a BIT.  Coordinate
    compression first, so the key space is O(n) rather than 10^9.
    """
    ranks = {v: i for i, v in enumerate(sorted(set(nums)))}
    bit = Fenwick(len(ranks))
    out = [0] * len(nums)
    for i in range(len(nums) - 1, -1, -1):
        r = ranks[nums[i]]
        out[i] = bit.prefix(r)                       # strictly smaller ranks
        bit.add(r, 1)
    return out

def get_modified_array(length, updates):
    """LC 370 - Range Addition.  The punchline: you need neither structure.

    All the updates arrive before any query, so a difference array does the
    whole job in O(n + k) with no tree at all.  A segment tree here is a
    correct answer to a question nobody asked.
    """
    d = [0] * (length + 1)
    for l, r, v in updates:
        d[l] += v
        d[r + 1] -= v
    out, run = [], 0
    for i in range(length):
        run += d[i]
        out.append(run)
    return out


na = NumArrayBIT([1, 3, 5])
print('307  sumRange(0, 2) =', na.sumRange(0, 2))
na.update(1, 2)
print('     after update(1, 2) =', na.sumRange(0, 2))
print('315  count_smaller([5, 2, 6, 1]) =', count_smaller([5, 2, 6, 1]))
print('370  get_modified_array(5, ...) =',
      get_modified_array(5, [[1, 3, 2], [2, 4, 3], [0, 2, -2]]))

307  sumRange(0, 2) = 9
     after update(1, 2) = 8
315  count_smaller([5, 2, 6, 1]) = [2, 1, 1, 0]
370  get_modified_array(5, ...) = [-2, 0, 3, 5, 3]


## 8. Tests

Every structure here is checked against brute force rather than reasoned about -
two of the bugs written while preparing this day (an off-by-one in the two-BIT
algebra, and a test that mutated its own reference array) were invisible to
reading and obvious to a random cross-check.

In [12]:
import random
rng = random.Random(29)
for _ in range(200):
    n = rng.randint(1, 40)
    a = [rng.randint(-9, 9) for _ in range(n)]

    seg = SegmentTree(list(a))
    fen = Fenwick(list(a))
    lazy = LazySegmentTree(list(a))
    rr = FenwickRangeRange(n)
    for i, v in enumerate(a):
        rr.range_add(i, i, v)
    ref = list(a)

    for _ in range(20):
        l = rng.randrange(n)
        r = rng.randrange(l + 1, n + 1)
        want = sum(ref[l:r])
        assert seg.query(l, r) == want, 'seg'
        assert fen.range_sum(l, r) == want, 'fenwick'
        assert lazy.range_sum(l, r - 1) == want, 'lazy'
        assert rr.range_sum(l, r - 1) == want, 'two-BIT'

        d = rng.randint(-5, 5)
        lazy.range_add(l, r - 1, d)
        rr.range_add(l, r - 1, d)
        for i in range(l, r):
            ref[i] += d
        for i in range(l, r):
            seg.update(i, ref[i])
            fen.add(i, d)

assert count_smaller([5, 2, 6, 1]) == [2, 1, 1, 0]
assert get_modified_array(5, [[1, 3, 2], [2, 4, 3], [0, 2, -2]]) == [-2, 0, 3, 5, 3]
print('all assertions passed')

all assertions passed


## Takeaways

- Wanting fast updates *and* fast range queries is what creates the whole family
  of structures; either goal alone is a one-liner.
- A segment tree answers any range with at most `2 * log2(n)` whole subtrees, and
  works for any associative operation because it never subtracts.
- Lazy propagation applies the same idea to updates: stop at the covered node,
  leave a note. Measured here, that is roughly 862x fewer node visits.
- A Fenwick tree is the same asymptotics in `n` cells and six lines, because
  `i & -i` encodes the tree implicitly - but it needs an operation with an
  inverse.
- Before reaching for either, check whether the updates and the queries are
  actually interleaved. If they are not, a difference array wins.